<h1 style="font-size:27px;line-height:1.3;font-weight:750;color:#173e60;margin:16px 0 12px 0;letter-spacing:-0.02em;">integration, cleaning and preprocessing</h1>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>We use <strong>only three sources</strong>: Bien’ici, PAP and ParuVendu. The original CSV files are never modified.</p>
<p>This notebook is organized using the stages of the <em>Fouille de données</em> course (chapter II, part 2):</p>
<ul>
<li><strong>Part A : Data integration :</strong> load three heterogeneous sources, align their columns and combine them.</li>
<li><strong>Part B : Data cleaning </strong> manage the four problems: inconsistent, redundant, noisy and missing data.</li>
<li><strong>Part C : Data preprocessing :</strong> calculate only documented derived attributes useful for the next analysis, then check and save.</li>
</ul>
<p><strong>EDA and visualization will be developed in a separate notebook.</strong> All exclusions and estimations are documented assumptions; estimated values are never presented as observed facts.</p>
</div>


<h2 style="font-size:21px;line-height:1.35;font-weight:700;color:#176395;padding:0 0 8px 0;border-bottom:2px solid #d5e9f6;margin:25px 0 14px 0;">Part A : Data integration</h2>


In [1]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd

# Required only for the optional Random Forest comparison.
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor

DATA_DIR = Path("marseille_scraping_outputs")
OUTPUT_DIR = Path("marseille_cleaning_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "bienici": DATA_DIR / "bienici.csv",
    "pap": DATA_DIR / "pap.csv",
    "paruvendu": DATA_DIR / "paruvendu.csv",
}

for site, path in FILES.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing {site} CSV: {path.resolve()}")
print("The three original CSV files were found.")

The three original CSV files were found.


<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">2. Load and inspect the source files</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>Postal codes are identifiers, so we read them as text rather than as quantities.</p>
</div>


In [2]:
raw_bienici = pd.read_csv(FILES["bienici"], dtype={"zipcode": "string"})
raw_pap = pd.read_csv(FILES["pap"], dtype={"zipcode": "string"})
raw_paruvendu = pd.read_csv(FILES["paruvendu"], dtype={"zipcode": "string"})

source_counts = pd.Series({
    "bienici": len(raw_bienici),
    "pap": len(raw_pap),
    "paruvendu": len(raw_paruvendu)
})
print("Original advertisements by website:")
print(source_counts)
print("Total collected:", int(source_counts.sum()))

Original advertisements by website:
bienici      685
pap           11
paruvendu    292
dtype: int64
Total collected: 988


<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">3. Harmonize the schemas</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>The websites use different column names. We rename equivalent fields, convert numeric fields, add columns that a source does not provide as <strong>missing</strong>, and preserve <code>source_site</code> and <code>listing_id</code> for traceability.</p>
</div>


In [3]:
COLUMNS = [
    "source_site", "listing_id", "url", "title", "property_type",
    "price_total_eur", "surface_m2", "rooms", "bedrooms", "zipcode",
    "charges_eur", "depot_garantie_eur", "honoraires_eur",
    "dpe_class", "ges_class", "is_furnished", "has_elevator",
    "has_balcony_terrace", "has_parking", "latitude", "longitude", "raw_text",
]
NUMERIC = [
    "price_total_eur", "surface_m2", "rooms", "bedrooms",
    "charges_eur", "depot_garantie_eur", "honoraires_eur",
    "latitude", "longitude"
]

def standardize_columns(table, site):
    result = table.rename(columns={
        "id": "listing_id",
        "dpe_energy_class": "dpe_class",
        "has_balcony": "has_balcony_terrace",
    }).reindex(columns=COLUMNS).copy()
    result["source_site"] = site
    for col in NUMERIC:
        result[col] = pd.to_numeric(result[col], errors="coerce")
    result["zipcode"] = result["zipcode"].astype("string")
    return result

bi = standardize_columns(raw_bienici, "bienici")
pap = standardize_columns(raw_pap, "pap")
pv = standardize_columns(raw_paruvendu, "paruvendu")
print("Shared number of columns:", len(COLUMNS))

Shared number of columns: 22


<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">4. Correct issues specific to each source</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<ul>
<li>Commercial premises are excluded <strong>only when their property type or the start of their title</strong> clearly identifies a non-residential listing. The description may mention an office in an otherwise residential apartment.</li>
<li>PAP charges have known extraction errors (the scraper sometimes copied the rent into the charges field). We mark suspicious charges as missing; we <strong>do not replace them with the rent</strong>.</li>
<li>Existing furnished status from ParuVendu is preserved; we never assume an unknown status means <code>Non</code>.</li>
</ul>
</div>


In [4]:
COMMERCIAL_TYPES = {
    "bureau", "bureaux", "commerce", "commerces", "local", "locaux",
    "atelier", "garage", "garages", "parking", "entrepot", "entrepôt",
    "boutique", "terrain"
}

def exclude_commercial(table, site):
    category = table["property_type"].fillna("").astype(str).str.casefold().str.strip()
    title = table["title"].fillna("").astype(str).str.casefold().str.strip()
    starts_commercial = title.str.contains(
        r"^(?:à louer\s+|location\s+(?:de\s+)?|louer\s+)?"
        r"(?:bureaux?\b|commerces?\b|(?:local|locaux)\b|atelier\b|garage\b|"
        r"parking\b|entrep[oô]t\b|boutique\b|terrain\b)",
        regex=True,
    )
    unwanted = category.isin(COMMERCIAL_TYPES) | starts_commercial
    print(site, "non-residential listings excluded:", int(unwanted.sum()))
    return table.loc[~unwanted].copy()

bi = exclude_commercial(bi, "Bien'ici")
pap = exclude_commercial(pap, "PAP")
pv = exclude_commercial(pv, "ParuVendu")

# The conservative PAP correction is based on an observed scraper problem.
pap_suspect = (
    (pap["charges_eur"] < 0)
    | (pap["charges_eur"] > 400)
    | (pap["charges_eur"] >= 0.45 * pap["price_total_eur"])
)
print("PAP charges flagged as unreliable:", int(pap_suspect.sum()))
pap.loc[pap_suspect, "charges_eur"] = np.nan

Bien'ici non-residential listings excluded: 1
PAP non-residential listings excluded: 0
ParuVendu non-residential listings excluded: 0
PAP charges flagged as unreliable: 9


<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">5. Combine the three sources</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>Use <code>pd.concat()</code> to stack listings vertically: there is no shared cross-platform ID for a database join. <code>source_site</code> always records the origin.</p>
</div>


In [5]:
df = pd.concat([bi, pap, pv], ignore_index=True)
collected_rows = len(df)
print("Rows after integration:", collected_rows)
print(df["source_site"].value_counts())
display(df.head(3))

Rows after integration: 987
source_site
bienici      684
paruvendu    292
pap           11
Name: count, dtype: int64


,source_site,listing_id,url,title,property_type,price_total_eur,surface_m2,rooms,bedrooms,zipcode,...,honoraires_eur,dpe_class,ges_class,is_furnished,has_elevator,has_balcony_terrace,has_parking,latitude,longitude,raw_text
0,bienici,hektor-xavierradisson13-203,NaN,Studio 26 rue Aramis 13014,flat,359.00,14.26,1.0,NaN,13014,...,187.00,C,B,Non,NaN,NaN,NaN,43.326823,5.386046,Contact : 06 80 12 05 66 . Situé 26 rue Aramis...
1,bienici,seiitra-elyade_bi-17290,NaN,NaN,flat,392.00,18.82,1.0,0.0,13009,...,246.91,A,A,Non,Oui,NaN,NaN,43.242241,5.432999,Résidence étudiante idéalement située à Marsei...
2,bienici,ag135528-554266103,NaN,T1,flat,407.92,13.00,1.0,NaN,13008,...,177.39,NS,NS,Non,NaN,NaN,NaN,43.267915,5.384386,"Découvrez cette chambre de bonne de 13,52 m², ..."


<h2 style="font-size:21px;line-height:1.35;font-weight:700;color:#176395;padding:0 0 8px 0;border-bottom:2px solid #d5e9f6;margin:25px 0 14px 0;">Part B : Data cleaning</h2>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>we separated cleaning into four tasks. Here they appear in a practical order: <strong>inconsistency → redundancy → noise → missing values</strong>. A problem can sometimes belong to more than one category.</p>
</div>

<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">6. Inconsistent data: standardize formats and check essential variables</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>Only valid Marseille postcodes <code>13001–13016</code> are kept. <code>13000</code> is not an individual arrondissement and is excluded rather than assigned arbitrarily. We also require a usable rent and surface; the limits €250–€10,000 and 9–500 m² are <strong>manual scope/quality filters</strong>, not statistical proof of errors.</p>
</div>


In [6]:
# A source may contain values such as "13006.0".
df["zipcode"] = df["zipcode"].astype("string").str.extract(r"(130\d{2})")[0]
valid_zipcodes = [f"130{i:02d}" for i in range(1, 17)]

before = len(df)
valid_rows = (
    df["zipcode"].isin(valid_zipcodes)
    & df["price_total_eur"].between(250, 10000)
    & df["surface_m2"].between(9, 500)
)
df = df.loc[valid_rows].copy()
print("Removed (postcode, rent or surface):", before - len(df))

# Validate DPE and GES: keep grades A–G only; invalid labels become missing.
for col in ["dpe_class", "ges_class"]:
    df[col] = df[col].astype("string").str.strip().str.upper()
    df[col] = df[col].where(df[col].isin(list("ABCDEFG")))

# Do not invent room counts using an arbitrary m²/room ratio.
df.loc[df["rooms"].le(0) | df["rooms"].gt(20), "rooms"] = np.nan
# Price per m² is needed for quality checks later.
df["price_per_m2"] = (df["price_total_eur"] / df["surface_m2"]).round(2)
print("Rows remaining:", len(df))

Removed (postcode, rent or surface): 38
Rows remaining: 949


<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">7. Redundant data: remove exact repeated identifiers</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>If the <strong>same website</strong> repeats the same nonempty URL or listing ID, keep the most complete occurrence. Listings with identical price, surface, rooms and postcode are <strong>review candidates</strong>, not confirmed duplicates. This avoids deleting two different apartments with similar characteristics.</p>
</div>


In [7]:
# Prefer the row containing more non-missing fields.
df = df.assign(_known=df.notna().sum(axis=1)).sort_values("_known", ascending=False)
df = df.drop(columns="_known")

has_url = df["url"].notna() & df["url"].astype(str).str.strip().ne("")
url_duplicate = has_url & df.duplicated(["source_site", "url"])
has_id = df["listing_id"].notna() & df["listing_id"].astype(str).str.strip().ne("")
id_duplicate = has_id & df.duplicated(["source_site", "listing_id"])
confirmed = url_duplicate | id_duplicate
print("Repeated URLs/IDs removed:", int(confirmed.sum()))
df = df.loc[~confirmed].copy()

# Compare similarities across sources without automatically removing them.
keys = ["zipcode", "price_total_eur", "surface_m2", "rooms"]
possible_duplicates = df.loc[df.duplicated(keys, keep=False)].sort_values(keys)
print("Rows with matching characteristics (review only):", len(possible_duplicates))
display(possible_duplicates[["source_site", "title", *keys]].head(10))

# Save this intermediate dataset before the more selective cleaning.
merged_rows = len(df)
merged_path = OUTPUT_DIR / "marseille_merged.csv"
df.to_csv(merged_path, index=False, encoding="utf-8-sig")
print("Saved merged data:", merged_path)

Repeated URLs/IDs removed: 0
Rows with matching characteristics (review only): 114


,source_site,title,zipcode,price_total_eur,surface_m2,rooms
45,bienici,Studio,13001,525.0,13.0,1.0
833,paruvendu,NaN,13001,525.0,13.0,1.0
85,bienici,Studio,13001,600.0,17.0,1.0
930,paruvendu,NaN,13001,600.0,17.0,1.0
205,bienici,Chambre 14m² - Quartier Notre-Dame-Du-Mont,13001,750.0,14.0,1.0
206,bienici,Chambre 14m² - YOLOCOLIVING,13001,750.0,14.0,1.0
225,bienici,"À Louer Petit T2 de 20,70m2 rénové avec clim...",13001,780.0,21.0,2.0
711,paruvendu,NaN,13001,780.0,21.0,2.0
285,bienici,Studio,13001,850.0,23.0,1.0
786,paruvendu,NaN,13001,850.0,23.0,1.0


Saved merged data: marseille_cleaning_outputs\marseille_merged.csv


<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">8. Noisy data: check short-term rentals and suspicious values</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>We remove seasonal rentals from Bien’ici and PAP, as well as listings with unusual prices per m². We don't use ParuVendu descriptions for this check because they may contain unrelated website text.
For incorrect charges or deposits, we keep the advertisement but replace the suspicious value with <code>NaN</code>.
The €8–€45/m², €400 charges and 45% charges-to-rent limits are <strong>project heuristics</strong>.</p>
</div>


In [8]:
clean = df.copy()
text = clean["raw_text"].fillna("").astype(str)

# Identify explicit short-term-rental wording in the description.
seasonal = text.str.contains(
    r"\blocation\s+saisonni[eè]re\b|\bpar\s+nuit\b|\bnuit[eé]e?s?\b",
    case=False, regex=True
)
not_seasonal = text.str.contains(
    r"\b(?:pas\s+de?|non)\s+location\s+saisonni[eè]re\b",
    case=False, regex=True
)
# Do not automatically filter ParuVendu using full-page text (menu contamination).
seasonal = seasonal & ~not_seasonal & clean["source_site"].isin(["bienici", "pap"])
in_range = clean["price_per_m2"].between(8, 45)
print("Explicit seasonal rentals excluded:", int(seasonal.sum()))
print("Outside €8–45 per m²:", int((~in_range).sum()))
before = len(clean)
clean = clean.loc[(~seasonal) & in_range].copy()
print("Total listings removed at this step:", before - len(clean))

# An impossible bedroom count becomes missing; no bedrooms are estimated here.
bad_bedrooms = (
    clean["bedrooms"].lt(0)
    | (clean["rooms"].notna() & clean["bedrooms"].gt(clean["rooms"]))
)
print("Suspicious bedroom values marked missing:", int(bad_bedrooms.sum()))
clean.loc[bad_bedrooms, "bedrooms"] = np.nan

# Financial consistency checks. An advertisement remains even if one value is invalid.
bad_charges = (
    (clean["charges_eur"] < 0)
    | (clean["charges_eur"] > 400)
    | (clean["charges_eur"] >= 0.45 * clean["price_total_eur"])
)
print("Suspicious charges marked missing:", int(bad_charges.sum()))
clean.loc[bad_charges, "charges_eur"] = np.nan

bad_deposit = (clean["depot_garantie_eur"] < 0) | (
    clean["depot_garantie_eur"] > 2.2 * clean["price_total_eur"]
)
bad_fees = (clean["honoraires_eur"] < 0) | (
    clean["honoraires_eur"] > 20 * clean["surface_m2"]
)
print("Suspicious deposits/fees marked missing:", int(bad_deposit.sum()), int(bad_fees.sum()))
clean.loc[bad_deposit, "depot_garantie_eur"] = np.nan
clean.loc[bad_fees & clean["source_site"].ne("pap"), "honoraires_eur"] = np.nan

Explicit seasonal rentals excluded: 2
Outside €8–45 per m²: 38
Total listings removed at this step: 40
Suspicious bedroom values marked missing: 4
Suspicious charges marked missing: 9
Suspicious deposits/fees marked missing: 2 5


<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">9. Complete missing parking information</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>Furnished status is already extracted during scraping, so we keep it unchanged.</p>
<p>We use regular expressions to find missing parking information in Bien’ici and PAP descriptions. We check negative statements and avoid confusing nearby parking with private parking.</p>
<p>If the information is unclear, we leave it missing.</p>
</div>


In [9]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    s = unicodedata.normalize("NFKD", str(value).casefold())
    return "".join(ch for ch in s if not unicodedata.combining(ch))


# Standardize Yes/No values from scraping.
def yes_no(value):
    s = normalize_text(value).strip()
    return {
        "oui": "Oui", "non": "Non",
        "true": "Oui", "false": "Non",
        "1": "Oui", "0": "Non"
    }.get(s, pd.NA)


for col in ["is_furnished", "has_elevator",
            "has_balcony_terrace", "has_parking"]:
    clean[col] = clean[col].apply(yes_no)


# Detect parking information in descriptions.
PARKING_POS = (
    r"\b(?:avec\s+(?:un\s+|une\s+)?(?:place\s+de\s+)?(?:parking|garage|box)"
    r"|place\s+de\s+parking\s+(?:privative?|incluse?|reservee?)"
    r"|garage\s+privatif|parking\s+(?:prive|privatif|inclus|reserve)"
    r"|box\s+(?:ferme|privatif))\b"
)

PARKING_NEG = (
    r"\b(?:sans|aucun(?:e)?|pas\s+de|absence\s+de)"
    r"\s+(?:place\s+de\s+)?(?:parking|garage|box)\b"
)

PARKING_NEAR = (
    r"\b(?:parking|garage|box)\s+"
    r"(?:a\s+proximite|proche|public|dans\s+la\s+rue)\b"
)


def parking_from_description(value):
    s = normalize_text(value)

    # Ignore parking mentioned only as nearby.
    s = re.sub(PARKING_NEAR, "", s)

    negative = bool(re.search(PARKING_NEG, s))
    positive = bool(re.search(
        PARKING_POS, re.sub(PARKING_NEG, "", s)
    ))

    if positive and negative:
        return pd.NA
    if negative:
        return "Non"
    if positive:
        return "Oui"
    return pd.NA


# Fill missing parking values only for Bien'ici and PAP.
safe_sites = clean["source_site"].isin(["bienici", "pap"])

parking_text = clean["raw_text"].apply(parking_from_description)

missing_parking = (
    clean["has_parking"].isna()
    & safe_sites
    & parking_text.notna()
)

clean.loc[missing_parking, "has_parking"] = parking_text[missing_parking]

print("Parking values filled:", missing_parking.sum())
print("\nFurnished status:")
print(clean["is_furnished"].value_counts(dropna=False))
print("\nParking status:")
print(clean["has_parking"].value_counts(dropna=False))


Parking values filled: 64

Furnished status:
is_furnished
Oui    440
NaN    247
Non    222
Name: count, dtype: int64

Parking status:
has_parking
<NA>    845
Oui      62
Non       2
Name: count, dtype: int64


<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">10. Missing data (Comparing Methods for Missing Charges: Median vs Random Forest)</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>We compare two methods to estimate missing monthly charges:</p>
<ul>
<li><strong>Median:</strong> estimates charges using the median charges per m² in each arrondissement.</li>
<li><strong>Random Forest:</strong> predicts charges using surface, rent, number of rooms, arrondissement and source website.</li>
</ul>
<p>We use <strong>5-fold cross-validation</strong> to evaluate both methods on the same data. Performance is measured using Mean Absolute Error (MAE)</p>
</div>

<div style="background-color:#eaf8f1;border-left:4px solid #22966e;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#1d4738;margin:7px 0 15px 0;">
<p>Random Forest achieves a lower average MAE (26.28 €) than the median method (39.26 €). Its errors also vary slightly less between folds.</p>
<p>Based on these results, we choose <strong>Random Forest</strong> to estimate missing charges. Known charges remain unchanged, and estimated values are identified in a separate column.</p>
</div>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>We do not estimate missing bedrooms, agency fees, deposits or DPE. Missing is acceptable when an estimate is not sufficiently justified.</p>
</div>


In [10]:

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline

# 1. Keep listings with known charges for comparison.
known = clean[clean["charges_eur"].notna() & (clean["surface_m2"] > 0)].copy()

num_cols = ["surface_m2", "price_total_eur", "rooms"]
cat_cols = ["zipcode", "source_site"]

X = known[num_cols + cat_cols].copy()
y = known["charges_eur"]

for col in cat_cols:
    X[col] = X[col].fillna("Unknown").astype(str)

# 2. Prepare Random Forest.
preprocessing = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

rf = make_pipeline(
    preprocessing,
    RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=3,
        random_state=42
    )
)

# 3. Compare both methods using the same 5 folds.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

median_errors = []
rf_errors = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Median charges per m², calculated from training data only.
    train_rates = y_train / X_train["surface_m2"]
    stats = train_rates.groupby(X_train["zipcode"]).agg(["count", "median"])
    local_medians = stats["median"].where(stats["count"] >= 5)

    rates = X_test["zipcode"].map(local_medians).fillna(train_rates.median())
    median_pred = X_test["surface_m2"] * rates

    median_errors.append(mean_absolute_error(y_test, median_pred))

    # Train and evaluate Random Forest on the same fold.
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)

    rf_errors.append(mean_absolute_error(y_test, rf_pred))

# 4. Display the comparison.
results = pd.DataFrame({
    "Median": median_errors,
    "Random Forest": rf_errors
})

print("Cross-validation MAE (lower is better):")
print(results.agg(["mean", "std"]).round(2))


# 5. Choose the method to fill missing charges.
CHARGES_METHOD = "random_forest"  # Or "median"

clean["charges_eur_estimated"] = False
missing = clean["charges_eur"].isna() & clean["surface_m2"].gt(0)

if missing.any():
    X_missing = clean.loc[missing, num_cols + cat_cols].copy()

    for col in cat_cols:
        X_missing[col] = X_missing[col].fillna("Unknown").astype(str)

    if CHARGES_METHOD == "median":
        # Calculate medians again using all known charges.
        all_rates = y / X["surface_m2"]
        stats = all_rates.groupby(X["zipcode"]).agg(["count", "median"])
        local_medians = stats["median"].where(stats["count"] >= 5)

        rates = X_missing["zipcode"].map(local_medians).fillna(all_rates.median())
        predictions = X_missing["surface_m2"] * rates

    elif CHARGES_METHOD == "random_forest":
        # Train on all known charges, then predict missing ones.
        rf.fit(X, y)
        predictions = pd.Series(rf.predict(X_missing), index=X_missing.index)

    else:
        raise ValueError("Choose 'median' or 'random_forest'.")

    # Keep only reasonable predictions.
    predictions = predictions.round()
    limit = np.minimum(400, 0.45 * clean.loc[missing, "price_total_eur"])

    valid = predictions.notna() & predictions.ge(0) & predictions.lt(limit)
    valid_rows = predictions.index[valid]

    clean.loc[valid_rows, "charges_eur"] = predictions.loc[valid_rows]
    clean.loc[valid_rows, "charges_eur_estimated"] = True

print("Method selected:", CHARGES_METHOD)
print("Listings with observed charges:", len(known))
print("Charges estimated:", clean["charges_eur_estimated"].sum())
print("Charges still missing:", clean["charges_eur"].isna().sum())

print("\nCharge statistics:")
display(clean["charges_eur"].describe().round(2))

print("\nEstimated charges by website:")
display(
    clean.groupby("source_site")["charges_eur_estimated"].sum()
)



Cross-validation MAE (lower is better):
      Median  Random Forest
mean   41.91          36.70
std     2.84           2.28
Method selected: random_forest
Listings with observed charges: 659
Charges estimated: 250
Charges still missing: 0

Charge statistics:


count    909.00
mean     100.64
std       66.32
min       10.00
25%       50.00
50%       80.00
75%      130.00
max      400.00
Name: charges_eur, dtype: float64


Estimated charges by website:


source_site
bienici       93
pap            9
paruvendu    148
Name: charges_eur_estimated, dtype: int64

<h2 style="font-size:21px;line-height:1.35;font-weight:700;color:#176395;padding:0 0 8px 0;border-bottom:2px solid #d5e9f6;margin:25px 0 14px 0;">Part C : Data preprocessing</h2>

<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">11. Derived numerical and housing variables</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p><code>price_per_m2</code> was already calculated for anomaly checks. <code>net_rent_eur</code> assumes the advertised total includes charges. Upfront moving cost is calculated <strong>only when all three observed/estimated components are available</strong>. DPE A–G is converted to an ordered numeric code without filling missing DPE values.</p>
</div>


In [12]:
clean["arrondissement"] = clean["zipcode"].str[-2:].astype(int)
clean["property_type_label"] = pd.cut(
    clean["rooms"], bins=[0, 1, 2, 3, np.inf],
    labels=["Studio / T1", "T2", "T3", "T4+"]
)
clean["surface_per_room"] = (clean["surface_m2"] / clean["rooms"]).round(2)
clean["charges_share_pct"] = (
    100 * clean["charges_eur"] / clean["price_total_eur"]
).round(1)


clean["net_rent_eur"] = (
    clean["price_total_eur"] - clean["charges_eur"]
).round(2)

# Unknown deposits and fees remain unknown: no 13 €/m² assumption.
clean["upfront_move_in_cost_eur"] = (
    clean["price_total_eur"] + clean["depot_garantie_eur"] + clean["honoraires_eur"]
).round(0)

DPE_MAP = {letter: i for i, letter in enumerate("ABCDEFG", start=1)}
clean["dpe_numeric"] = clean["dpe_class"].map(DPE_MAP)
clean["dpe_is_missing"] = clean["dpe_class"].isna().astype(int)

<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">12. Geographic zones and distance</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<ol>
<li><strong>Five exploratory geographic groups</strong>, defined below from the 16 arrondissements. They are <strong>not official socio-economic classes</strong> or evidence that one group is richer than another. The labels are descriptive.</li>
<li>A <strong>separate two-group comparison</strong> uses <em>7e–8e</em> versus <em>14e–16e</em>. This is a focused contrast, not the same as the five-zone classification.</li>
</ol>
<p>Distance to the Vieux-Port is <strong>straight-line distance, not travel time</strong>. We calculate it only where the website provided numeric GPS coordinates; we do <strong>not impute coordinates using the postcode median</strong>, which would invent a false location.</p>
</div>


In [13]:

def urban_zone(arr):
    if arr in [7, 8]:
        return "1. Littoral sud (7e, 8e)"
    if arr in [6, 9, 12]:
        return "2. Sud et Est (6e, 9e, 12e)"
    if arr in [1, 4, 5]:
        return "3. Centre et Timone (1er, 4e, 5e)"
    if arr in [2, 10, 11, 13]:
        return "4. Euromed et Est (2e, 10e, 11e, 13e)"
    return "5. Nord et 3e (3e, 14e, 15e, 16e)"

clean["urban_zone"] = clean["arrondissement"].apply(urban_zone)
print("Count by exploratory geographic group:")
print(clean["urban_zone"].value_counts().sort_index())

clean["gps_was_missing"] = clean[["latitude", "longitude"]].isna().any(axis=1)
clean["has_exact_gps"] = (~clean["gps_was_missing"]).astype(int)  # Legacy name for analysis compatibility.

# A basic geographic sanity check before calculating distance.
valid_gps = (
    clean["latitude"].between(43.15, 43.43)
    & clean["longitude"].between(5.25, 5.65)
)
lat0, lon0 = np.radians([43.2951, 5.3744])
lat, lon = np.radians(clean["latitude"]), np.radians(clean["longitude"])
a = np.sin((lat-lat0)/2)**2 + np.cos(lat0)*np.cos(lat)*np.sin((lon-lon0)/2)**2
clean["dist_vieux_port_km"] = (
    6371 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
).where(valid_gps).round(2)
print("Listings with a usable reported GPS:", int(valid_gps.sum()))

Count by exploratory geographic group:
urban_zone
1. Littoral sud (7e, 8e)                 182
2. Sud et Est (6e, 9e, 12e)              204
3. Centre et Timone (1er, 4e, 5e)        194
4. Euromed et Est (2e, 10e, 11e, 13e)    231
5. Nord et 3e (3e, 14e, 15e, 16e)         98
Name: count, dtype: int64
Listings with a usable reported GPS: 641


<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">13. Equipment indicators and amenity score</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>We convert equipment information into numeric values:</p>
<ul>
<li>Yes = 1</li>
<li>No = 0</li>
<li>Unknown = missing</li>
</ul>
<p>The <code>amenity_score</code> counts positive features among five characteristics: furnished, elevator, balcony/terrace, parking, and renovation.</p>
<p><strong>Score formula:</strong></p>
<p>$$
\text{Amenity Score} = F + E + B + P + R
$$</p>
<p>Where:</p>
<ul>
<li>$F$: Furnished</li>
<li>$E$: Elevator</li>
<li>$B$: Balcony or terrace</li>
<li>$P$: Parking</li>
<li>$R$: Renovated</li>
</ul>
<p>Each confirmed positive feature adds <strong>1 point</strong>, giving a score from <strong>0 to 5</strong>. Missing values do not add points.</p>
<p>We also calculate <code>amenities_known_count</code> to record how many features have known values.</p>
</div>


In [14]:
# Convert structured statuses to 1 / 0 / missing (never fill missing with zero).
for col in ["is_furnished", "has_elevator", "has_balcony_terrace", "has_parking"]:
    clean[col + "_binary"] = clean[col].map({"Oui": 1, "Non": 0}).astype(float)

RENOVATION_POS = r"\b(?:renove\w*|refait\s+a\s+neuf|entierement\s+renove\w*)\b"
RENOVATION_NEG = r"\b(?:non\s+renove\w*|pas\s+renove\w*|a\s+renover|travaux\s+a\s+prevoir)\b"

def renovation_from_description(value):
    s = normalize_text(value)
    neg = bool(re.search(RENOVATION_NEG, s))
    pos = bool(re.search(RENOVATION_POS, re.sub(RENOVATION_NEG, "", s)))
    if pos and neg:
        return np.nan
    if neg:
        return 0.0
    if pos:
        return 1.0
    return np.nan

clean["is_renovated"] = clean["raw_text"].apply(renovation_from_description)
# ParuVendu raw_text contains whole-page navigation, so text indicators are not trustworthy there.
clean.loc[clean["source_site"].eq("paruvendu"), "is_renovated"] = np.nan
# The older analysis expects the has_parking column in numeric form.
clean["has_parking"] = clean["has_parking_binary"]

feature_columns = [
    "is_furnished_binary", "has_elevator_binary", "has_balcony_terrace_binary",
    "has_parking", "is_renovated"
]
clean["amenities_known_count"] = clean[feature_columns].notna().sum(axis=1)
clean["amenity_score"] = clean[feature_columns].sum(axis=1, min_count=1)

print("Known feature counts (0 to 5):")
print(clean["amenities_known_count"].value_counts().sort_index())
display(clean[["source_site", "is_furnished", "is_furnished_binary",
               "has_parking", "amenities_known_count", "amenity_score"]].head(20))

Known feature counts (0 to 5):
amenities_known_count
0    156
1    307
2    297
3    123
4     26
Name: count, dtype: int64


,source_site,is_furnished,is_furnished_binary,has_parking,amenities_known_count,amenity_score
467,bienici,Non,0.0,NaN,3,2.0
430,bienici,Non,0.0,NaN,3,0.0
90,bienici,Non,0.0,NaN,3,0.0
367,bienici,Oui,1.0,NaN,3,2.0
598,bienici,Non,0.0,NaN,3,2.0
112,bienici,Non,0.0,NaN,3,2.0
107,bienici,Non,0.0,NaN,3,1.0
378,bienici,Non,0.0,1.0,4,3.0
360,bienici,Non,0.0,NaN,3,2.0
600,bienici,Non,0.0,NaN,3,2.0


<h3 style="font-size:17px;line-height:1.45;font-weight:650;color:#2471a3;margin:21px 0 10px 0;">16. Final quality checks and export</h3>

<div style="background-color:#eef6fc;border-left:4px solid #3498db;padding:11px 15px;border-radius:5px;font-size:13px;line-height:1.65;color:#243746;margin:7px 0 15px 0;">
<p>Report row counts, missing data and estimated charges. Save the merged dataset (after integration and basic checks) and the final cleaned/preprocessed dataset. We do not change the three original CSVs.</p>
</div>


In [15]:
clean = clean.sort_values(["zipcode", "price_total_eur"]).reset_index(drop=True)
ready_path = OUTPUT_DIR / "marseille_analysis_ready.csv"
clean.to_csv(ready_path, index=False, encoding="utf-8-sig")

print("Original total:", int(source_counts.sum()))
print("After integration:", collected_rows)
print("After basic validity and deduplication:", merged_rows)
print("Final rows:", len(clean))
print("\nFinal records by website:")
print(clean["source_site"].value_counts())
print("\nMissing main fields:")
print(clean[["price_total_eur", "surface_m2", "rooms", "charges_eur",
             "bedrooms", "honoraires_eur", "depot_garantie_eur", "is_furnished", "dpe_class"]].isna().sum())
print("\nCharges estimated:", int(clean["charges_eur_estimated"].sum()))
print("\nSaved:", merged_path, "and", ready_path)
display(clean.head(5))

Original total: 988
After integration: 987
After basic validity and deduplication: 949
Final rows: 909

Final records by website:
source_site
bienici      641
paruvendu    257
pap           11
Name: count, dtype: int64

Missing main fields:
price_total_eur         0
surface_m2              0
rooms                   0
charges_eur             0
bedrooms              258
honoraires_eur        321
depot_garantie_eur    222
is_furnished          247
dpe_class             119
dtype: int64

Charges estimated: 250

Saved: marseille_cleaning_outputs\marseille_merged.csv and marseille_cleaning_outputs\marseille_analysis_ready.csv


,source_site,listing_id,url,title,property_type,price_total_eur,surface_m2,rooms,bedrooms,zipcode,...,gps_was_missing,has_exact_gps,dist_vieux_port_km,is_furnished_binary,has_elevator_binary,has_balcony_terrace_binary,has_parking_binary,is_renovated,amenities_known_count,amenity_score
0,bienici,ag132391-556779443,NaN,Appartement,flat,440.0,19.00,1.0,0.0,13001,...,False,1,0.39,0.0,NaN,NaN,NaN,NaN,1,0.0
1,paruvendu,NaN,https://www.paruvendu.fr/immobilier/location/a...,NaN,appartement,450.0,11.00,1.0,NaN,13001,...,True,0,NaN,1.0,NaN,NaN,NaN,NaN,1,1.0
2,bienici,hektor-projetimgest-2400,NaN,"STUDIO 17,84 M2 PROCHE DU VIEUX PORT ET DE LA ...",flat,480.0,17.84,1.0,NaN,13001,...,False,1,0.81,1.0,NaN,NaN,NaN,NaN,1,1.0
3,bienici,hektor-sogebrimo-7902,NaN,T1 Meublé Rue Lafayette 13001,flat,500.0,18.00,1.0,NaN,13001,...,False,1,0.88,1.0,0.0,NaN,NaN,NaN,2,1.0
4,bienici,ag130161-556171982,NaN,Studio,flat,525.0,13.00,1.0,0.0,13001,...,False,1,0.80,NaN,1.0,NaN,NaN,1.0,2,2.0
